In [1]:
# Cell 1: Imports & basic config

import os
import sys
import numpy as np
import pandas as pd

import torch
import torch.backends.cudnn as cudnn

# Disable CuDNN RNNs to avoid weird backward issues with LSTMs
cudnn.enabled = False

import lightning.pytorch as pl
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping

from pytorch_forecasting import TimeSeriesDataSet
from pytorch_forecasting.models import TemporalFusionTransformer as TFTModel
from pytorch_forecasting.metrics import MAE

# Set a seed for reproducibility
pl.seed_everything(42, workers=True)

print("Torch version      :", torch.__version__)
print("Lightning version  :", pl.__version__)

# 👉 Path to your parquet
DATA_PATH = r"C:\Users\admin\Desktop\Forex_algo\code\Data\EUR_USD_H1.parquet"
print("Data path:", DATA_PATH)


c:\Users\admin\Desktop\Forex_algo\code\venv\Lib\site-packages\pytorch_forecasting\models\base\_base_model.py:28: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm
Seed set to 42


Torch version      : 2.5.1+cu121
Lightning version  : 2.5.6
Data path: C:\Users\admin\Desktop\Forex_algo\code\Data\EUR_USD_H1.parquet


In [2]:
# Cell 2: Load Parquet and inspect

df = pd.read_parquet(DATA_PATH)
print("Raw shape:", df.shape)
print("Columns  :", df.columns.tolist())

# Ensure time is datetime
df["time"] = pd.to_datetime(df["time"])

# Sort by time
df = df.sort_values("time").reset_index(drop=True)

df.head()


Raw shape: (61473, 14)
Columns  : ['time', 'volume', 'mid_o', 'mid_h', 'mid_l', 'mid_c', 'bid_o', 'bid_h', 'bid_l', 'bid_c', 'ask_o', 'ask_h', 'ask_l', 'ask_c']


,time,volume,mid_o,mid_h,mid_l,mid_c,bid_o,bid_h,bid_l,bid_c,ask_o,ask_h,ask_l,ask_c
0,2016-01-07 00:00:00+00:00,542,1.07764,1.07832,1.07744,1.07778,1.07757,1.07823,1.07735,1.07770,1.07772,1.07840,1.07752,1.07787
1,2016-01-07 01:00:00+00:00,3167,1.07776,1.08100,1.07748,1.08029,1.07768,1.08092,1.07740,1.08020,1.07784,1.08109,1.07756,1.08038
2,2016-01-07 02:00:00+00:00,1567,1.08026,1.08176,1.07996,1.08152,1.08018,1.08169,1.07987,1.08144,1.08035,1.08184,1.08005,1.08159
3,2016-01-07 03:00:00+00:00,914,1.08156,1.08257,1.08150,1.08187,1.08147,1.08249,1.08142,1.08178,1.08164,1.08265,1.08157,1.08196
4,2016-01-07 04:00:00+00:00,649,1.08190,1.08256,1.08156,1.08236,1.08182,1.08247,1.08147,1.08228,1.08199,1.08264,1.08163,1.08245


In [3]:
# Cell 3: Basic structure (close, time features, series_id, time_idx)

# We'll use mid close as main price
df["close"] = df["mid_c"]

# Categorical time features as strings
df["hour"] = df["time"].dt.hour.astype(str)
df["day_of_week"] = df["time"].dt.dayofweek.astype(str)

# Single series id
df["series_id"] = "eurusd"

# Time index as integer
df["time_idx"] = np.arange(len(df), dtype=int)

print(df[["time", "close", "hour", "day_of_week", "series_id", "time_idx"]].head())


                       time    close hour day_of_week series_id  time_idx
0 2016-01-07 00:00:00+00:00  1.07778    0           3    eurusd         0
1 2016-01-07 01:00:00+00:00  1.08029    1           3    eurusd         1
2 2016-01-07 02:00:00+00:00  1.08152    2           3    eurusd         2
3 2016-01-07 03:00:00+00:00  1.08187    3           3    eurusd         3
4 2016-01-07 04:00:00+00:00  1.08236    4           3    eurusd         4


In [4]:
# Cell 4: Add technical indicators (from your project) + engineered features + target_return
from technicals.indicators import add_standard_indicators
# Make sure Python sees your Forex_algo\code folder where indicators.py lives.
# If you run this notebook from that folder already, this is safe anyway.
project_root = r"C:\Users\admin\Desktop\Forex_algo\code"
if project_root not in sys.path:
    sys.path.append(project_root)



print("Using indicators from:", project_root)

# 1) Apply all your indicators + aliases
df = add_standard_indicators(df)

# 2) Engineered features
df["momentum_oc"] = df["mid_o"] - df["mid_c"]
df["avg_price_hl"] = (df["mid_h"] + df["mid_l"]) / 2.0
df["range_hl"] = df["mid_h"] - df["mid_l"]
df["typical_price_ohlc"] = (df["mid_o"] + df["mid_h"] + df["mid_l"] + df["mid_c"]) / 4.0

# Log volume
df["log_volume"] = np.log1p(df["volume"])

# 3) Target: next-hour log-return on close
df["target_return"] = np.log(df["close"].shift(-1)) - np.log(df["close"])

# Drop NaNs from rolling / shift
df = df.dropna().reset_index(drop=True)

print("Shape after indicators & target:", df.shape)
df.head()

Using indicators from: C:\Users\admin\Desktop\Forex_algo\code
Shape after indicators & target: (61273, 48)


,time,volume,mid_o,mid_h,mid_l,mid_c,bid_o,bid_h,bid_l,bid_c,...,ema_5,ema_20,ema_50,ema_200,momentum_oc,avg_price_hl,range_hl,typical_price_ohlc,log_volume,target_return
0,2016-01-19 07:00:00+00:00,1298,1.08736,1.08764,1.08595,1.08652,1.08726,1.08756,1.08587,1.08645,...,1.087710,1.088764,1.089161,1.088334,0.00084,1.086795,0.00169,1.086867,7.169350,0.001784
1,2016-01-19 08:00:00+00:00,1450,1.08654,1.08846,1.08636,1.08846,1.08647,1.08832,1.08629,1.08832,...,1.087960,1.088735,1.089133,1.088336,-0.00192,1.087410,0.00210,1.087455,7.280008,-0.001121
2,2016-01-19 09:00:00+00:00,1626,1.08844,1.08890,1.08702,1.08724,1.08834,1.08883,1.08694,1.08699,...,1.087720,1.088593,1.089059,1.088323,0.00120,1.087960,0.00188,1.087900,7.394493,0.000055
3,2016-01-19 10:00:00+00:00,1376,1.08728,1.08834,1.08662,1.08730,1.08702,1.08827,1.08655,1.08722,...,1.087580,1.088470,1.088990,1.088312,-0.00002,1.087480,0.00172,1.087385,7.227662,-0.000920
4,2016-01-19 11:00:00+00:00,941,1.08728,1.08732,1.08599,1.08630,1.08721,1.08726,1.08593,1.08623,...,1.087153,1.088263,1.088885,1.088289,0.00098,1.086655,0.00133,1.086723,6.848005,0.001067


In [5]:
# NEW Cell 5: Manually curated FEATURE_COLS_EXT

# Just to check what numeric columns exist (optional print)
num_cols = df.select_dtypes(include=["number", "float", "int"]).columns.tolist()
print("All numeric columns:", num_cols)

# Curated feature list:
FEATURE_COLS_EXT = [
    # Core price + volume
    "mid_o", "mid_h", "mid_l", "mid_c",
    "volume", "log_volume",

    # Your standard indicators (from add_standard_indicators)
    "rsi",
    "macd", "macd_signal", "macd_hist",
    "atr14",
    "bb_lower", "bb_middle", "bb_upper",
    "ema_5", "ema_20", "ema_50", "ema_200",

    # Engineered features
    "momentum_oc", "avg_price_hl", "range_hl", "typical_price_ohlc",
]

print("\nFEATURE_COLS_EXT (curated):")
print(FEATURE_COLS_EXT)


All numeric columns: ['volume', 'mid_o', 'mid_h', 'mid_l', 'mid_c', 'bid_o', 'bid_h', 'bid_l', 'bid_c', 'ask_o', 'ask_h', 'ask_l', 'ask_c', 'close', 'time_idx', 'BB_MA', 'BB_UP', 'BB_LW', 'ATR_14', 'EMA', 'KeUp', 'KeLo', 'RSI_14', 'MACD', 'SIGNAL', 'HIST', 'rsi', 'macd', 'macd_signal', 'macd_hist', 'bb_middle', 'bb_upper', 'bb_lower', 'atr14', 'ema_5', 'ema_20', 'ema_50', 'ema_200', 'momentum_oc', 'avg_price_hl', 'range_hl', 'typical_price_ohlc', 'log_volume', 'target_return']

FEATURE_COLS_EXT (curated):
['mid_o', 'mid_h', 'mid_l', 'mid_c', 'volume', 'log_volume', 'rsi', 'macd', 'macd_signal', 'macd_hist', 'atr14', 'bb_lower', 'bb_middle', 'bb_upper', 'ema_5', 'ema_20', 'ema_50', 'ema_200', 'momentum_oc', 'avg_price_hl', 'range_hl', 'typical_price_ohlc']


In [6]:
# Cell 6: Time-based train / val / test split

df = df.sort_values("time_idx").reset_index(drop=True)

N = len(df)
train_end = int(N * 0.7)
val_end   = int(N * 0.85)

train_df = df.iloc[:train_end].copy()
val_df   = df.iloc[train_end:val_end].copy()
test_df  = df.iloc[val_end:].copy()

print("Total samples:", N)
print("Train:", len(train_df), "Val:", len(val_df), "Test:", len(test_df))
print("time_idx ranges:")
print(" train:", train_df["time_idx"].min(), "->", train_df["time_idx"].max())
print(" val  :", val_df["time_idx"].min(),   "->", val_df["time_idx"].max())
print(" test :", test_df["time_idx"].min(),  "->", test_df["time_idx"].max())


Total samples: 61273
Train: 42891 Val: 9191 Test: 9191
time_idx ranges:
 train: 199 -> 43089
 val  : 43090 -> 52280
 test : 52281 -> 61471


In [7]:
# Cell 7: TimeSeriesDataSet definitions

MAX_ENCODER_LENGTH = 168   # 7 days of H1
MAX_PREDICTION_LENGTH = 1
BATCH_SIZE = 1280           # If you get OOM, reduce to 128 or 64

training = TimeSeriesDataSet(
    train_df,
    time_idx="time_idx",
    target="target_return",
    group_ids=["series_id"],

    max_encoder_length=MAX_ENCODER_LENGTH,
    max_prediction_length=MAX_PREDICTION_LENGTH,

    time_varying_unknown_reals=FEATURE_COLS_EXT,
    time_varying_known_categoricals=["hour", "day_of_week"],
    static_categoricals=["series_id"],

    target_normalizer=None,        # keep raw log-returns (compatible with live engine)
    allow_missing_timesteps=True,
    add_relative_time_idx=True,
    add_encoder_length=True,
    add_target_scales=False,
)

validation = TimeSeriesDataSet.from_dataset(
    training,
    data=val_df,
    stop_randomization=True,
)

test = TimeSeriesDataSet.from_dataset(
    training,
    data=test_df,
    stop_randomization=True,
)

print("Number of samples:")
print(" training   :", len(training))
print(" validation :", len(validation))
print(" test       :", len(test))


Number of samples:
 training   : 42723
 validation : 9023
 test       : 9023


In [8]:
# Cell 8: Dataloaders from TimeSeriesDataSet

train_loader = training.to_dataloader(
    train=True,
    batch_size=BATCH_SIZE,
    num_workers=0,
)

val_loader = validation.to_dataloader(
    train=False,
    batch_size=BATCH_SIZE,
    num_workers=0,
)

test_loader = test.to_dataloader(
    train=False,
    batch_size=BATCH_SIZE,
    num_workers=0,
)

print("Batches:")
print(" train batches:", len(train_loader))
print(" val batches  :", len(val_loader))
print(" test batches :", len(test_loader))


Batches:
 train batches: 33
 val batches  : 8
 test batches : 8


In [9]:
# Cell 9: Build the TFT model (v3 rich-feature version)

hidden_size = 32
attention_heads = 4
dropout = 0.10
hidden_continuous_size = 16
lstm_layers = 2
LEARNING_RATE = 1e-3

tft = TFTModel.from_dataset(
    training,
    learning_rate=LEARNING_RATE,

    hidden_size=hidden_size,
    lstm_layers=lstm_layers,
    attention_head_size=attention_heads,
    dropout=dropout,
    hidden_continuous_size=hidden_continuous_size,

    loss=MAE(),          # MAE on target_return
    output_size=1,       # 1-step regression

    log_interval=50,
    log_val_interval=1,
    reduce_on_plateau_patience=4,
)

print("TFT model created.")
print("Number of parameters:", sum(p.numel() for p in tft.parameters()))


TFT model created.
Number of parameters: 128752


c:\Users\admin\Desktop\Forex_algo\code\venv\Lib\site-packages\lightning\pytorch\utilities\parsing.py:210: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
c:\Users\admin\Desktop\Forex_algo\code\venv\Lib\site-packages\lightning\pytorch\utilities\parsing.py:210: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.


In [10]:
# Cell 10: Trainer configuration

checkpoint_cb = ModelCheckpoint(
    monitor="val_loss",
    mode="min",
    save_top_k=1,
    filename="eurusd_h1_tft_v3_rich-{epoch:02d}-{val_loss:.6f}",
)

earlystop_cb = EarlyStopping(
    monitor="val_loss",
    mode="min",
    patience=8,
    min_delta=1e-5,
)

accelerator = "gpu" if torch.cuda.is_available() else "cpu"

trainer = pl.Trainer(
    max_epochs=80,
    accelerator=accelerator,
    devices=1,
    gradient_clip_val=0.1,
    precision=32,                     # keep float32 to avoid half-precision issues
    callbacks=[checkpoint_cb, earlystop_cb],
    log_every_n_steps=20,
)

trainer


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores


In [11]:
# Cell 11: Training

trainer.fit(
    model=tft,
    train_dataloaders=train_loader,
    val_dataloaders=val_loader,
)

print("Best checkpoint path:", checkpoint_cb.best_model_path)


You are using a CUDA device ('NVIDIA GeForce RTX 4070 Laptop GPU') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

   | Name                               | Type                            | Params | Mode 
------------------------------------------------------------------------------------------------
0  | loss                               | MAE                             | 0      | train
1  | logging_metrics                    | ModuleList                      | 0      | train
2  | input_embeddings                   | MultiEmbedding                  | 241    | train
3  | prescalers                         | ModuleDict                      | 768    | train
4  | static_variable_selection

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

c:\Users\admin\Desktop\Forex_algo\code\venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:433: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=21` in the `DataLoader` to improve performance.


c:\Users\admin\Desktop\Forex_algo\code\venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:433: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=21` in the `DataLoader` to improve performance.


Epoch 24: 100%|██████████| 33/33 [03:48<00:00,  0.14it/s, v_num=53, train_loss_step=0.00242, val_loss=0.00241, train_loss_epoch=0.00325]
Best checkpoint path: c:\Users\admin\Desktop\Forex_algo\code\lightning_logs\version_53\checkpoints\eurusd_h1_tft_v3_rich-epoch=16-val_loss=0.001043.ckpt


In [12]:
# Cell 12: Evaluation on test set (MAE + direction accuracy)

ckpt_path = checkpoint_cb.best_model_path
print("Using checkpoint:", ckpt_path)

best_model = TFTModel.load_from_checkpoint(ckpt_path)

# Predictions on test set
preds_raw = best_model.predict(test_loader)      # tensor [N, 1]
preds = preds_raw.detach().cpu().reshape(-1)

# True targets
all_targets = []
for batch in test_loader:
    x, y_tuple = batch          # (x, (y, weight))
    y = y_tuple[0]
    all_targets.append(y.detach().cpu().reshape(-1))

targets = torch.cat(all_targets)

preds_np = preds.numpy()
targets_np = targets.numpy()

# Metrics
mae = np.mean(np.abs(preds_np - targets_np))
print(f"Test MAE (log-return units): {mae:.8f}")

pred_dir = np.sign(preds_np)
true_dir = np.sign(targets_np)
dir_acc = (pred_dir == true_dir).mean()
print(f"Direction accuracy: {dir_acc:.4f} ({dir_acc*100:.2f}%)")

print("\nSample comparison (first 5):")
for i in range(5):
    print(f"{i}: pred={preds_np[i]:+0.6f}, true={targets_np[i]:+0.6f}")


Using checkpoint: c:\Users\admin\Desktop\Forex_algo\code\lightning_logs\version_53\checkpoints\eurusd_h1_tft_v3_rich-epoch=16-val_loss=0.001043.ckpt


c:\Users\admin\Desktop\Forex_algo\code\venv\Lib\site-packages\lightning\pytorch\utilities\parsing.py:210: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
c:\Users\admin\Desktop\Forex_algo\code\venv\Lib\site-packages\lightning\pytorch\utilities\parsing.py:210: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
c:\Users\admin\Desktop\Forex_algo\code\venv\Lib\site-packages\lightning\pytorch\trainer\c

Test MAE (log-return units): 0.00099965
Direction accuracy: 0.4948 (49.48%)

Sample comparison (first 5):
0: pred=+0.000643, true=-0.000130
1: pred=+0.000984, true=-0.000084
2: pred=+0.000564, true=-0.000363
3: pred=+0.000971, true=+0.000354
4: pred=+0.000901, true=+0.000196


In [13]:
# Cell 13: Copy best checkpoint to a stable filename for the live bot

import shutil

# Example: put the model one level above Data, next to your code
model_dst = os.path.abspath(
    os.path.join(os.path.dirname(DATA_PATH), "..", "tft_ret_rich_v5.ckpt")
)

print("Source checkpoint:", ckpt_path)
print("Destination      :", model_dst)

shutil.copy2(ckpt_path, model_dst)
print("Checkpoint copied.")


Source checkpoint: c:\Users\admin\Desktop\Forex_algo\code\lightning_logs\version_53\checkpoints\eurusd_h1_tft_v3_rich-epoch=16-val_loss=0.001043.ckpt
Destination      : C:\Users\admin\Desktop\Forex_algo\code\tft_ret_rich_v5.ckpt
Checkpoint copied.
